[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [ ]:
#This is a mathematical trick that turn the attention complexity from O(S^2 * D) to O(S * D^2)

#In general if instead of doing Q @ K.T (S^2) and then @ V (D) we do
# Q @ (K.T @ V) we'd achieve O(S * D^2), maintaining the same numerical result, thanks to the associative rule (Q @ K.T) @ V  = Q @ (K.T @ V)
#The problem is that with SOFTMAX BEFORE doin @ V the associative rule is not in place anymore and we cannot use this trick
#For this, we have to substitute the SOFTMAX with another method to weight the sum on the values
#The idea is to use a SEPARABLE kernel such that phi(Q @ K.T) = phi(Q) @ phi(K.T) (which softmax is not) so that we can separate them and perform phi(Q) @ (phi(K.T) @ V)
#as we wanted to do
#this kernel is the elu function + 1. This works because elu(x) = x if x > 0 and e^x - 1 if x < 0, so with elu(x) + 1 we have x + 1 if x > 0 and e^x if x < 0
#which is ALWAYS POSITIVE --> they emulate PROBABILITIES (like softmax did)
#finally, we need to normalize these weights s.t. their sum is 1 (like softmax), so we place at the denominator sum_keys(phi(Q) @ phi(K.T)) (in formula above it is Q.T just because there is a colomn-wise convention while in code we consider the row-wise convention)
#and since the phi function (elu) is separable and the sum is over keys, we can bring out the Q, achieving the denominator in the .md cell above.
#--> instead of computing weight (Qj * Ki) and then summing them (denominator softmax like) we can first sum the keys and then multiply for Q, it is the same numerically.
#but in this way we do just one sum to be shared across all queries

def linear_attention(Q, K, V):
    """
    Q: (B, S_q, d_q)
    K: (B, S_k, d_k)
    V: (B, S_k, d_v)

    return: (B, S_q, d_v)
    """

    def kernel(x):
      return F.elu(x) + 1

    inner = kernel(K).transpose(-2, -1) @ V #(B, d_k, d_v)
    outer = kernel(Q) @ inner #(B, S_q, d_v)

    key_sum = torch.sum(K, dim = 1).unsqueeze(-1) #sum all keys to obtain (B, d_k, 1)
    den = Q @ key_sum #(B, S_q, d_q) @ (B, d_k, 1) with d_k == d_q --> (B, S_q, 1), correspond to sum(Q @ K.T)

    return outer / den

In [11]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [12]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.5ms)
  ✅ [2/4] No NaN or Inf (3.6ms)
  ✅ [3/4] Gradient flow (1.7ms)
  ✅ [4/4] Runs fast on long sequences (linear complexity) (26.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (33.1ms total)
  Progress saved. Run status() to see your dashboard.

